In [1]:
import pandas as pd

data = {
    "Review": [
        "This movie was absolutely fantastic",
        "I loved the acting and story",
        "Best movie I have ever seen",
        "The film was boring and slow",
        "I hated this movie",
        "Worst movie of the year",
        "Amazing performance and direction",
        "Terrible plot and bad acting",
        "It was a wonderful experience",
        "Not worth watching at all"
    ],
    "Sentiment": [
        "positive",
        "positive",
        "positive",
        "negative",
        "negative",
        "negative",
        "positive",
        "negative",
        "positive",
        "negative"
    ]
}

df = pd.DataFrame(data)
df.to_csv("movie_reviews.csv", index=False)

print("Dataset created successfully: movie_reviews.csv")
print(df.head())



Dataset created successfully: movie_reviews.csv
                                Review Sentiment
0  This movie was absolutely fantastic  positive
1         I loved the acting and story  positive
2          Best movie I have ever seen  positive
3         The film was boring and slow  negative
4                   I hated this movie  negative


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,GRU, Input,Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences 

In [3]:
df=pd.read_csv("movie_reviews.csv")


In [8]:
# =====================================
# Movie Review Sentiment Analysis using GRU
# =====================================

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Input
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load dataset
df = pd.read_csv("movie_reviews.csv")

# Convert text to lowercase
texts = df["Review"].str.lower()

# Extract labels
labels = df["Sentiment"]

# Encode labels (positive=1, negative=0)
le = LabelEncoder()
y = le.fit_transform(labels)

# Tokenize text
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)

X_seq = tokenizer.texts_to_sequences(texts)

# Padding
max_len = max(len(seq) for seq in X_seq)

X_padded = pad_sequences(X_seq, maxlen=max_len, padding='post')

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_padded, y,
    test_size=0.2,
    random_state=42
)

# Build GRU model
model = Sequential([
    
    Input(shape=(max_len,)),
    # Input shape = sentence length

    Embedding(
        input_dim=len(tokenizer.word_index)+1,
        output_dim=16
    ),
    # Convert words into dense vectors

    GRU(64),
    # 64 GRU units
    # Learns sequential dependencies

    Dense(32, activation="relu"),
    # Hidden layer

    Dense(1, activation="sigmoid")
    # Binary output (positive / negative)
])

# Compile model
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Train model
model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=2,
    validation_split=0.2,
    verbose=1
)

print("\nGRU Sentiment Model Trained Successfully")

# Evaluate
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:", accuracy)


Epoch 1/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 253ms/step - accuracy: 0.5000 - loss: 0.6927 - val_accuracy: 0.5000 - val_loss: 0.6909
Epoch 2/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.7708 - loss: 0.6836 - val_accuracy: 0.5000 - val_loss: 0.6947
Epoch 3/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.6458 - loss: 0.6761 - val_accuracy: 0.5000 - val_loss: 0.6976
Epoch 4/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.4583 - loss: 0.6855 - val_accuracy: 0.5000 - val_loss: 0.6998
Epoch 5/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.8333 - loss: 0.6475 - val_accuracy: 0.5000 - val_loss: 0.7033
Epoch 6/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.7708 - loss: 0.6385 - val_accuracy: 0.5000 - val_loss: 0.7072
Epoch 7/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.4583 - loss: 0.6747 - val_accuracy: 0.5000 - val_loss: 0.7113
Epoch 8/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.6458 - loss: 0.6398 - val_accuracy: 0.5000 - val_loss

In [9]:
def predict_sentiment(text):
    text = text.lower()

    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_len, padding='post')

    prediction = model.predict(padded, verbose=0)[0][0]

    return "positive" if prediction > 0.5 else "negative"

# Test
sample_review = "amazing"
print("Review:", sample_review)
print("Predicted Sentiment:", predict_sentiment(sample_review))


Review: amazing
Predicted Sentiment: negative
